# CLEVER retrieval + repair on SageMaker (Lean + Mathlib + Ollama)

**Instance:** a GPU instance, e.g. **ml.g4dn.xlarge** (T4, 16 GB GPU), with **at least 50 GB storage**
(notebook instance: *Volume size* = 50 GB; Studio JupyterLab space: *Storage* = 50 GB). Default 5 GB is too small.
Kernel: `conda_python3` (notebook instance) or the default Python 3 (Studio).

**Everything lives on the persistent disk** (Lean, Mathlib, the model, logs, cache), so after stopping and
starting the instance just run cells 1–3 again (they take ~1 minute when everything is installed), then the
experiment cell you were on. Finished problems are skipped and model replies are cached.

Ollama runs in its own process session: **interrupting a cell does not stop it**, and cell 3 restarts it if needed.

**Stop the instance when you are done** — it is billed per hour while running.

## 0. GPU + disk check
Must show a GPU (Tesla T4) and at least ~25 GB free.

In [1]:
import os
WORK = '/home/ec2-user/SageMaker' if os.path.isdir('/home/ec2-user/SageMaker') else os.path.expanduser('~')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h {WORK} | tail -1

name, memory.total [MiB]
Tesla T4, 15360 MiB
/dev/nvme2n1     20G  182M   20G   1% /home/sagemaker-user


In [2]:
!rm -rf ~/.cache/pip && df -h ~ | tail -1


/dev/nvme2n1     20G  189M   20G   1% /home/sagemaker-user


In [3]:
!rm -rf ~/.cache/mathlib && df -h ~ | tail -1


/dev/nvme2n1     20G  189M   20G   1% /home/sagemaker-user


## 1. Paths, code, Python environment
A venv on the persistent disk holds `clever-bench` (and with it the Lean project + Mathlib), so nothing is lost on restart.
Torch is installed CPU-only: it is only used for retrieval embeddings, and it saves ~2 GB of disk.

In [4]:
import os, sys, subprocess, shutil
WORK = '/home/ec2-user/SageMaker' if os.path.isdir('/home/ec2-user/SageMaker') else os.path.expanduser('~')
REPO = f'{WORK}/Clever'
VENV = f'{WORK}/venv'
PY = f'{VENV}/bin/python'
os.environ['ELAN_HOME'] = f'{WORK}/.elan'
os.environ['OLLAMA_MODELS'] = f'{WORK}/ollama-models'
os.environ['PATH'] = f'{WORK}/.elan/bin:' + os.environ['PATH']

if not os.path.isdir(REPO):
    !git clone -q https://github.com/Aditi-Nandiraju/Clever.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}

if not os.path.exists(PY):
    !{sys.executable} -m venv {VENV}
    !{PY} -m pip install -q --upgrade pip
    !{PY} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!{PY} -m pip install -q -r requirements.txt
print('WORK =', WORK, '| python =', PY)

/home/sagemaker-user/Clever
WORK = /home/sagemaker-user | python = /home/sagemaker-user/venv/bin/python


## 2. Lean 4 + Mathlib (~10–20 min the first time, seconds afterwards)
Only `Imports.AllImports` is built: every problem file imports just that.
The last line compiles a tiny test file: it must print `4` and no errors.

In [5]:
if not os.path.exists(f"{os.environ['ELAN_HOME']}/bin/lake"):
    !curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none --no-modify-path

LEAN_DIR = subprocess.check_output(
    [PY, '-c', 'from clever_bench.benchmark import get_clever_lean_project_path as g; print(g())'], text=True).strip()
print('Lean project:', LEAN_DIR)
!cd {LEAN_DIR} && lake exe cache get
!cd {LEAN_DIR} && lake build Imports.AllImports

os.makedirs(f'{LEAN_DIR}/temp', exist_ok=True)
open(f'{LEAN_DIR}/temp/sanity.lean', 'w').write('import Imports.AllImports\n#eval (2 + 2 : Nat)\nexample : (1 : Nat) + 1 = 2 := by norm_num\n')
!cd {LEAN_DIR} && lake lean temp/sanity.lean

info: downloading installer
Lean project: /home/sagemaker-user/venv/lib/python3.12/site-packages/clever_bench/lean4
info: downloading https://releases.lean-lang.org/lean4/v4.24.0/lean-4.24.0-linux.tar.zst
438.3 MiB / 438.3 MiB (100 %) 327.6 MiB/s ETA:   0 s
info: installing /home/sagemaker-user/.elan/toolchains/leanprover--lean4---v4.24.0
info: mathlib: cloning https://github.com/leanprover-community/mathlib4
info: mathlib: checking out revision 'f897ebcf72cd16f89ab4577d0c826cd14afaafc7'
info: plausible: cloning https://github.com/leanprover-community/plausible
info: plausible: checking out revision 'dfd06ebfe8d0e8fa7faba9cb5e5a2e74e7bd2805'
info: LeanSearchClient: cloning https://github.com/leanprover-community/LeanSearchClient
info: LeanSearchClient: checking out revision '99657ad92e23804e279f77ea6dbdeebaa1317b98'
info: importGraph: cloning https://github.com/leanprover-community/import-graph
info: importGraph: checking out revision 'd768126816be17600904726ca7976b185786e6b9'
info: pr

## 3. Ollama on the GPU (run this after every restart)
Installs Ollama if missing, starts it detached from the notebook, and keeps the model on the persistent disk.
The last line must show **100% GPU** under PROCESSOR.

In [6]:
import time, urllib.request

if not shutil.which('ollama'):
    !(sudo apt-get -qq update && sudo apt-get -qq install -y zstd pciutils) >/dev/null 2>&1 || sudo yum install -y -q zstd pciutils >/dev/null 2>&1 || true
    !curl -fsSL https://ollama.com/install.sh | sh
    # the installer may start a system service that stores models on the small root disk: stop it, we run our own
    !sudo systemctl stop ollama >/dev/null 2>&1; sudo systemctl disable ollama >/dev/null 2>&1; true

def ollama_up():
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=3)
        return True
    except Exception:
        return False

os.makedirs(os.environ['OLLAMA_MODELS'], exist_ok=True)
if not ollama_up():
    subprocess.Popen(['ollama', 'serve'], stdout=open(f'{WORK}/ollama.log', 'a'),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
print('ollama running:', ollama_up())

open(f'{WORK}/Modelfile', 'w').write('FROM qwen2.5:7b\nPARAMETER num_ctx 32768\n')
!ollama pull qwen2.5:7b && ollama create qwen2.5-32k -f {WORK}/Modelfile
!ollama run qwen2.5-32k "hi" > /dev/null; ollama ps

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ollama running: True
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠦ pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏  71 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   4% ▕                  ▏ 179 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   5% ▕                  ▏ 237 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   8% ▕█                 ▏ 351 MB/4.7 GB                  pulling manifest 
p

In [7]:
!df -h ~ | tail -1


/dev/nvme2n1     20G   16G  4.7G  77% /home/sagemaker-user


In [8]:
!rm -rf ~/.cache/mathlib ~/.cache/pip && df -h ~ | tail -1


/dev/nvme2n1     20G   15G  5.3G  74% /home/sagemaker-user


## 4. Retrieval corpus (~2 min, only needed once)

In [9]:
if not os.path.exists('data/corpus.jsonl'):
    !{PY} corpus.py
else:
    print('corpus already built')

wrote /home/sagemaker-user/Clever/data/corpus.jsonl: 13180 lemmas, 8 examples


## 5. Quick test: 3 dev problems (~10–25 min)
Stages: `parse` = answer not in the required format, `impl` = implementation fails to compile or a test,
`proof` = implementation OK but proof fails, `ok PROVED` = success.

In [10]:
!{PY} run_experiment.py --configs C1_baseline --split dev --limit 3 --seeds 0


== C1_baseline seed 0 (dev): 3 problems to run
  C1_baseline s0 p0   a1: parse
  C1_baseline s0 p11  a1: impl
  C1_baseline s0 p0   a2: impl
  C1_baseline s0 p11  a2: impl
  C1_baseline s0 p0   a3: impl
  C1_baseline s0 p11  a3: impl
  C1_baseline s0 p0   a4: parse
  C1_baseline s0 p11  a4: impl
  C1_baseline s0 p0   a5: impl
  C1_baseline s0 p11  a5: impl
  C1_baseline s0 p17  a1: impl
  C1_baseline s0 p17  a2: impl
  C1_baseline s0 p17  a3: impl
  C1_baseline s0 p17  a4: parse
  C1_baseline s0 p17  a5: impl
== C1_baseline seed 0 done in 17.0 min


### Why did attempts fail?
Real Lean errors about the code (`unknown identifier`, `type mismatch`, failed tests) = the model is weak; that is a valid result.
`Timeout`, `object file ... does not exist` or errors mentioning `Imports` = Lean setup problem: rerun cell 2.

In [11]:
import json, glob, collections
rows = [json.loads(l) for f in glob.glob('logs/*_dev.jsonl') for l in open(f)]
print(len(rows), 'attempts:', collections.Counter(r['stage'] for r in rows))
for stage in ['impl', 'proof', 'parse']:
    for r in [r for r in rows if r['stage'] == stage][:2]:
        print(f"\n===== {stage}  {r['config']} p{r['pid']} a{r['attempt']}  (llm {r['sec_llm']}s, total {r['sec_total']}s)")
        print((r['feedback'] or '')[:500])

15 attempts: Counter({'impl': 12, 'parse': 3})

===== impl  C1_baseline p11 a1  (llm 133.0s, total 170.9s)
error at line 42 `let result := String.mk len fun i => if a.get (String.Pos.mk i) = b.get (String.Pos.mk i) then '0' else '1'`: Application type mismatch: The argument
  len
has type
  ℕ
but is expected to have type
  List Char
in the application
  { data := len }

error at line 47 `#test implementation "010" "110" = "100"`: aborting evaluation since the expression depends on the 'sorry' axiom, which can lead to runtime instability and crashes.

To attempt to evaluate anyway despite the risks, u

===== impl  C1_baseline p0 a2  (llm 133.5s, total 148.9s)
error at line 39 `| _, _ when found => found`: Invalid pattern: Expected an identifier in function position, but found
  _

error at line 56 `#test implementation ([1, 2, 3]: List Rat) 0.5 = false`: aborting evaluation since the expression depends on the 'sorry' axiom, which can lead to runtime instability and crashes.

To attempt 

## 6. Baseline vs repair on the 20 dev problems (~1–1.5 h)
Safe to interrupt and rerun: it resumes.

In [ ]:
!{PY} run_experiment.py --configs C1_baseline C2_repair --split dev --seeds 0

In [16]:
!{PY} run_experiment.py --configs C1_baseline C2_repair --limit 20 --seeds 0
!{PY} run_experiment.py --configs C3_retrieval C4_both --limit 20 --seeds 0



== C1_baseline seed 0 (test): 20 problems to run
  C1_baseline s0 p2   a1: impl
  C1_baseline s0 p3   a1: proof
  C1_baseline s0 p2   a2: proof
  C1_baseline s0 p2   a3: impl
  C1_baseline s0 p3   a2: proof
  C1_baseline s0 p2   a4: proof
  C1_baseline s0 p3   a3: proof
  C1_baseline s0 p2   a5: proof
  C1_baseline s0 p3   a4: proof
  C1_baseline s0 p3   a5: impl
^C
Batches: 100%|███████████████████████████████| 104/104 [03:13<00:00,  1.86s/it]
retriever: dense:sentence-transformers/all-MiniLM-L6-v2

== C3_retrieval seed 0 (test): 20 problems to run
^C


In [ ]:
!{PY} run_experiment.py --configs C2_repair --limit 4 --seeds 0


== C2_repair seed 0 (test): 4 problems to run


In [ ]:
!{PY} run_experiment.py --configs C3_retrieval --limit 4 --seeds 0

## 7. Results (dev)

In [ ]:
!{PY} analyze.py --split dev
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/dev/summary.csv'))
display(pd.read_csv('results/dev/errors.csv'))
display(Image('results/dev/curve.png'))

## 8. Add retrieval (C3, C4) on the dev problems (~1.5–2 h), then rerun cell 7

In [ ]:
!{PY} run_experiment.py --configs C3_retrieval C4_both --split dev --seeds 0

## 9. Final numbers: 40 test problems
Freeze `config.py` and `prompts.py` first. Long; rerun after any interruption, it resumes.
Results: `results/test/` inside the repo folder.

In [ ]:
!{PY} run_experiment.py --configs C1_baseline C2_repair --seeds 0
!{PY} run_experiment.py --configs C3_retrieval C4_both --seeds 0
!{PY} analyze.py
import pandas as pd
display(pd.read_csv('results/test/summary.csv'))